# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Гончар Александр Сергеевич
**Файлы данных:** `data/clients.csv`, `data/transactions.csv`

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "data/clients.csv"
RAW_TXN = "data/transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

Пробуем прочитать
файл с разными параметрами. Неверная кодировка дает `UnicodeDecodeError`,
неверный разделитель дает один столбец вместо нескольких.

In [2]:
for path in [RAW_CLIENTS, RAW_TXN]:
    for enc in ["utf-8", "cp1251"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                print(path, enc, repr(sep), "- столбцов:", df.shape[1])
            except UnicodeDecodeError:
                print(path, enc, repr(sep), "- ошибка кодировки")

data/clients.csv utf-8 ',' - столбцов: 16
data/clients.csv utf-8 ';' - столбцов: 1
data/clients.csv cp1251 ',' - ошибка кодировки
data/clients.csv cp1251 ';' - ошибка кодировки
data/transactions.csv utf-8 ',' - ошибка кодировки
data/transactions.csv utf-8 ';' - ошибка кодировки


data/transactions.csv cp1251 ',' - столбцов: 1


data/transactions.csv cp1251 ';' - столбцов: 9


`clients.csv` кодировка UTF-8, разделитель ',' (16 столбцов). `transactions.csv` кодировка cp1251,
разделитель ';' (9 столбцов). Но то, что файл открылся в cp1251, еще не значит, что кодировка верная так как
в cp1251 откроется почти любой файл, просто вместо букв может получиться мусор.
Поэтому дальше будем смотреть на сам текст.

Смотрим первые строки transactions и проверяем, что русские слова читаются.

In [3]:
pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", nrows=5)

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990102,C003921,2023-11-24 09:00:10,"1018,07",RUB,Кафе/рестораны,5812,Карта,успешно
1,990012,C002226,2023-11-25 17:53:53,"454,49",RUB,Продукты,5411,Карта,успешно
2,990027,C000544,2023-11-27 13:30:37,"1194,49",RUB,Продукты,5411,Онлайн-банк,успешно
3,990169,C002702,2023-11-27,"755,99",RUB,Онлайн-сервисы,5815,Карта,успешно
4,990082,C001351,2023-12-12,"1911,83",RUB,Продукты,5411,Карта,успешно


Категории, каналы и статусы читаются как нормальные русские слова,
значит кодировка cp1251 выбрана верно.
Также в amount дробная часть отделена запятой.

Читаем `transactions.csv` без указания десятичного разделителя и смотрим тип amount.

In [4]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";")
print(txn_try["amount"].dtype, txn_try["amount"].head(3).tolist())

str ['1018,07', '454,49', '1194,49']


Видим, что amount прочитался как текст, так как pandas по умолчанию ждет точку, а в файле запятая. Поэтому указываем `decimal=","`.

In [5]:
txn_try = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", decimal=",")
print(txn_try["amount"].dtype, txn_try["amount"].head(5).tolist())

float64 [1018.07, 454.49, 1194.49, 755.99, 1911.83]


С `decimal=","` amount стал числом. pandas делает столбец числовым только если смог перевести каждое значение, так что все суммы прочитались правильно.

Итоговое чтение обоих файлов с найденными параметрами.

`client_id`, `txn_id`, `branch_code`, `mcc_code` читаем как текст. Это метки, а не числа и складывать их бессмысленно. У `branch_code` вида '00409' при чтении числом потерялись бы ведущие нули.
Даты pandas сам оставляет текстом, форматы дат в файлах разные, в разделе 3.1 этим займемся.

In [6]:
clients_raw = pd.read_csv(RAW_CLIENTS, dtype={"client_id": str, "branch_code": str})
txn_raw = pd.read_csv(RAW_TXN, encoding="cp1251", sep=";", decimal=",",
                      dtype={"txn_id": str, "client_id": str, "mcc_code": str})

Проверяем результат: `info()` показывает число строк, тип каждого столбца и сколько в нем непустых значений.

In [7]:
clients_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 5690 entries, 0 to 5689
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   client_id           5690 non-null   str    
 1   birth_date          5690 non-null   str    
 2   gender              5690 non-null   str    
 3   city                5690 non-null   str    
 4   segment             5690 non-null   str    
 5   loyalty_tier        5690 non-null   str    
 6   card_type           5690 non-null   str    
 7   registration_date   5690 non-null   str    
 8   card_open_date      5690 non-null   str    
 9   income_source       5690 non-null   str    
 10  monthly_income      5204 non-null   float64
 11  has_mobile_app      5690 non-null   str    
 12  app_last_login      3525 non-null   str    
 13  satisfaction_score  2331 non-null   float64
 14  branch_code         5690 non-null   str    
 15  marketing_opt_in    5690 non-null   str    
dtypes: float64(2), st

In [8]:
txn_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 626169 entries, 0 to 626168
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   txn_id        626169 non-null  str    
 1   client_id     626169 non-null  str    
 2   txn_datetime  626169 non-null  str    
 3   amount        626169 non-null  float64
 4   currency      626169 non-null  str    
 5   category      626169 non-null  str    
 6   mcc_code      626169 non-null  str    
 7   channel       626169 non-null  str    
 8   status        626169 non-null  str    
dtypes: float64(1), str(8)
memory usage: 43.0 MB


Итог загрузки:

`clients.csv` – 5690 строк и 16 столбцов, `transactions.csv` – 626169 строк и 9 столбцов.

Столбцы `amount` и `monthly_income` прочитались как числа (`float64`), а id и даты остались текстом. По `Non-Null Count` видно пропуски в `monthly_income`, `app_last_login` и `satisfaction_score`, их разберем в разделе 3.3.

`satisfaction_score` тоже получился `float64`, хотя оценки целые от 1 до 5, это из-за пропусков. 

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

Собираем короткий паспорт обеих таблиц: объем, типы, ключ и единица наблюдения, то есть что означает одна строка.
По описанию в `clients.csv` одна строка – это один клиент с ключом `client_id`, а в `transactions.csv` одна строка – одна операция с ключом `txn_id`.
Если так, ключ должен быть уникальным. Доля лишних строк считается от числа строк таблицы.

In [9]:
passport = pd.DataFrame({
    "строк": [len(clients_raw), len(txn_raw)],
    "столбцов": [clients_raw.shape[1], txn_raw.shape[1]],
    "числовых столбцов": [clients_raw.select_dtypes("number").shape[1], txn_raw.select_dtypes("number").shape[1]],
    "ключ": ["client_id", "txn_id"],
    "уникальных ключей": [clients_raw["client_id"].nunique(), txn_raw["txn_id"].nunique()],
}, index=["clients.csv", "transactions.csv"])
passport["лишних строк"] = passport["строк"] - passport["уникальных ключей"]
passport["доля лишних, %"] = (passport["лишних строк"] / passport["строк"] * 100).round(1)
passport

,строк,столбцов,числовых столбцов,ключ,уникальных ключей,лишних строк,"доля лишних, %"
clients.csv,5690,16,2,client_id,5635,55,1.0
transactions.csv,626169,9,1,txn_id,624669,1500,0.2


В `clients.csv` 16 столбцов, из них числовых только 2 (`monthly_income` и `satisfaction_score`), в `transactions.csv` из 9 столбцов числовой один – `amount`. Остальное текст.

Ключи не уникальны. В `clients.csv` 5690 строк, но разных `client_id` только 5635, лишних строк 55.
В `transactions.csv` лишних строк 1500.
Значит один клиент или одна операция могут встречаться несколько раз, и единица наблюдения не совпадает с описанием.

### 2.2. Дубли

Начинаем с `clients.csv`. Повтор `client_id` бывает двух видов: полная копия строки (все поля совпадают) или разные записи с одним id.
Первое – техническое дублирование выгрузки, второе – противоречие: непонятно, какая запись верная.

In [10]:
print("лишних строк по client_id:", clients_raw["client_id"].duplicated().sum(), "из", len(clients_raw))
print("из них полных копий строк:", clients_raw.duplicated().sum())

лишних строк по client_id: 55 из 5690
из них полных копий строк: 40


Из 55 повторов 40 – точные копии, их можно просто убрать. А вот у 15 клиентов записи разные, дальше посмотрим, какие поля не совпадают.

In [11]:
clients_dup = clients_raw[clients_raw["client_id"].duplicated(keep=False)]
clients_dup_unique = clients_dup.drop_duplicates()
clients_conflict = clients_dup_unique[clients_dup_unique["client_id"].duplicated(keep=False)]
print("client_id с разными данными в повторах:", clients_conflict["client_id"].nunique())

diff_cols = (clients_conflict.groupby("client_id").nunique(dropna=False) > 1).sum()
diff_cols[diff_cols > 0]

client_id с разными данными в повторах: 15


city       14
segment    10
dtype: int64

Различия только в двух полях: `city` (14 клиентов) и `segment` (10 клиентов). По числам не понять, это опечатки в написании города или разные записи, поэтому смотрим на примеры. 

In [12]:
clients_conflict.sort_values("client_id")[["client_id", "city", "segment"]]

,client_id,city,segment
3322,C001083,Новосибирск,Премиальный
2067,C001083,Новосибирск,Зарплатный
3937,C001632,ЕКАТЕРИНБУРГ,Private
4864,C001632,Санкт-Петербург,Массовый
4711,C002094,Новосибирск,Зарплатный
786,C002094,г.Екатеринбург,Премиальный
1955,C002355,Нижний Новгород,Премиальный
4763,C002355,Казань,Массовый
4369,C002475,г.Екатеринбург,Премиальный
1601,C002475,Казань,Зарплатный


Названия городов в данных записаны по-разному: один и тот же город встречается с разным регистром, сокращениями и с «г.» или без. Даже если считать разные написания одним городом, у большинства из 15 клиентов две записи все равно указывают на разные города, а часто и на разные сегменты. Для этих клиентов настоящие город и сегмент неизвестны.

Один человек может быть записан под разными `client_id`. Проверяем по признакам, которые у человека не меняются: дата рождения, пол, дата прихода в банк и дата открытия карты. Берем по одной строке на `client_id`, чтобы уже найденные повторы не мешали.

In [13]:
stable = ["birth_date", "gender", "registration_date", "card_open_date"]
one_per_id = clients_raw.drop_duplicates("client_id")
print("одних и тех же людей под разными client_id:", one_per_id.duplicated(subset=stable).sum())

одних и тех же людей под разными client_id: 0


Под разными `client_id` нет клиентов с одинаковыми датой рождения, полом и датами регистрации и открытия карты.

Теперь `transactions.csv`, проверяем повторы `txn_id` так же.

In [14]:
print("лишних строк по txn_id:", txn_raw["txn_id"].duplicated().sum(), "из", len(txn_raw))
print("из них полных копий строк:", txn_raw.duplicated().sum())

лишних строк по txn_id: 1500 из 626169


из них полных копий строк: 1259


Из 1500 повторов 1259 – точные копии строк. У остальных 241 записи с одним `txn_id` чем-то отличаются, посмотрим чем они будут отличаться.

In [15]:
txn_dup = txn_raw[txn_raw["txn_id"].duplicated(keep=False)]
txn_dup_unique = txn_dup.drop_duplicates()
txn_conflict = txn_dup_unique[txn_dup_unique["txn_id"].duplicated(keep=False)]
print("txn_id с разными данными в повторах:", txn_conflict["txn_id"].nunique())

txn_diff_cols = (txn_conflict.groupby("txn_id").nunique(dropna=False) > 1).sum()
txn_diff_cols[txn_diff_cols > 0]

txn_id с разными данными в повторах: 241


txn_datetime    241
dtype: int64

Различается только по полю `txn_datetime`. Проверяем, как именно они различаются.

In [16]:
pairs = txn_conflict.groupby("txn_id")["txn_datetime"]
print("пар, где дата одна и та же:", (pairs.agg(lambda s: s.str[:10].nunique()) == 1).sum())
print("пар, где в одной записи только дата без времени:", (pairs.agg(lambda s: s.str.len().min()) == 10).sum())
txn_conflict.sort_values("txn_id").head(4)

пар, где дата одна и та же: 241
пар, где в одной записи только дата без времени: 241


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
208,100202,C003891,2024-01-01 11:56:08,495.24,RUB,Продукты,5411,Карта,успешно
209,100202,C003891,2024-01-01,495.24,RUB,Продукты,5411,Карта,успешно
3673,103657,C002052,2024-01-05,783.06,RUB,Продукты,5411,Карта,успешно
3674,103657,C002052,2024-01-05 14:51:41,783.06,RUB,Продукты,5411,Карта,успешно


Во всех 241 парах дата одна и та же, просто в одной записи время есть, а в другой только дата. Остальные поля, включая сумму, совпадают. Значит это одна и та же операция, записанная дважды. У нас выходит 1259 точных копий и 241 копия, которые отличаются только форматом даты.

### 2.3. Соответствие данных описанию полей из задания

Сначала категориальные поля. Для тех, у которых в описании задания перечислены значения, проверяем, нет ли в данных других значений. Считаем по сырым строкам, дубли еще не убраны.

In [17]:
expected = {
    "segment": ["Массовый", "Зарплатный", "Премиальный", "Private"],
    "loyalty_tier": ["Базовый", "Серебро", "Золото", "Платина"],
    "card_type": ["Дебетовая", "Кредитная", "Премиальная"],
}
for col, values in expected.items():
    extra = clients_raw.loc[~clients_raw[col].isin(values), col]
    print(col, "| значений не из описания:", len(extra))

expected_txn = {
    "channel": ["Карта", "Токен", "СБП", "Онлайн-банк"],
    "status": ["успешно", "отклонено", "возврат"],
}
for col, values in expected_txn.items():
    extra = txn_raw.loc[~txn_raw[col].isin(values), col]
    print(col, "| значений не из описания:", len(extra))

segment | значений не из описания: 0
loyalty_tier | значений не из описания: 0
card_type | значений не из описания: 0
channel | значений не из описания: 0
status | значений не из описания: 0


В `segment`, `loyalty_tier`, `card_type`, `channel` и `status` все значения совпадают с описанием.

Для остальных полей значения в описании не перечислены, поэтому смотрим на них через `value_counts()`.

In [18]:
for col in ["gender", "has_mobile_app", "marketing_opt_in", "income_source"]:
    display(clients_raw[col].value_counts(dropna=False))

gender
Ж    2859
М    2831
Name: count, dtype: int64

has_mobile_app
Да     3525
Нет    2165
Name: count, dtype: int64

marketing_opt_in
Да     5022
Нет     668
Name: count, dtype: int64

income_source
Зарплата в нашем банке     2547
Зарплата в другом банке    1599
Не указан                  1544
Name: count, dtype: int64

`gender`, `has_mobile_app` и `marketing_opt_in` принимают по два значения, тут все чисто.
В `income_source` есть значение `Не указан` – 1544 строки из 5690. Это пропуск, просто вместо пустой ячейки в ней стоит текст, поэтому `info()` считает эти строки заполненными, хотя источник дохода в них неизвестен.

В описании сказано, что `category` – одна из девяти категорий. Проверяем.

In [19]:
txn_raw["category"].value_counts()

category
Продукты               164934
Транспорт               65912
Одежда и обувь          65275
АЗС                     58023
Кафе и рестораны        51038
Развлечения и досуг     48162
Аптеки                  45391
Онлайн-сервисы          44112
Связь и интернет        29305
Связь и Интернет        19592
Кафе/рестораны          17226
кафе и рестораны        17199
Name: count, dtype: int64

Категорий 12, а не 9. Лишние появились из-за разного написания. Если их объединить, то у нас получится как раз 9.

Про валюту в описании ничего не сказано, смотрим, какие значения есть.

In [20]:
txn_raw["currency"].value_counts()

currency
RUB    623636
EUR      1283
USD      1250
Name: count, dtype: int64

Кроме рублей есть операции в `EUR` и `USD`. Суммы в разных валютах нельзя просто складывать, а курсов в данных нет.

По описанию `city` – город проживания. Смотрим, как записаны города. 

In [21]:
print("разных значений city:", clients_raw["city"].nunique())
clients_raw["city"].value_counts()

разных значений city: 17


city
Москва                1129
Новосибирск            681
Санкт-Петербург        657
Казань                 563
Нижний Новгород        534
Екатеринбург           526
г. Новосибирск         224
МОСКВА                 183
г. Москва              178
 москва                171
Н. Новгород            162
казань                 142
г. Санкт-Петербург     131
Санкт Петербург        113
СПб                    109
г.Екатеринбург          98
ЕКАТЕРИНБУРГ            89
Name: count, dtype: int64

Разных значений 17, а городов на самом деле 6 .Один и тот же город записан по-разному: разный регистр, приставка, пробел в начале, сокращения, запись без дефиса.

По описанию `mcc_code` соответствует категории. Смотрим таблицу сопряженности: сколько операций в каждой паре категории и кода.

In [22]:
pd.crosstab(txn_raw["category"], txn_raw["mcc_code"])

mcc_code,4111,4814,5411,5541,5651,5812,5815,5912,7832
category,,,,,,,,,
АЗС,29,20,23,57854,22,24,15,17,19
Аптеки,18,15,7,19,11,14,21,45277,9
Кафе и рестораны,12,12,18,16,18,50913,22,10,17
Кафе/рестораны,3,9,7,5,4,17184,8,3,3
Одежда и обувь,22,24,33,17,65106,18,22,17,16
Онлайн-сервисы,11,9,14,18,16,8,44013,10,13
Продукты,53,61,164482,53,58,53,66,57,51
Развлечения и досуг,20,15,16,16,19,14,8,14,48040
Связь и Интернет,8,19540,4,8,8,7,7,7,3


У каждой категории есть свой основной код, в нем почти все операции, а разные написания одной категории имеют один и тот же код. Но в каждой строке есть небольшие числа и в чужих столбцах – это операции, где код не совпадает с категорией. Считаем, сколько их.

In [23]:
main_mcc = txn_raw.groupby("category")["mcc_code"].agg(lambda s: s.mode()[0])
mcc_mismatch = txn_raw["mcc_code"] != txn_raw["category"].map(main_mcc)
print("операций, где mcc_code не совпадает с основным кодом категории:", mcc_mismatch.sum(), "из", len(txn_raw))

операций, где mcc_code не совпадает с основным кодом категории: 1645 из 626169


Таких операций 1645 из 626169. Это расхождение описания с фактом: код не всегда соответствует категории. По самим данным нельзя сказать, что из двух записано неверно – категория или код.

Теперь числовые поля. Начинаем с суммы операции `amount`.

In [24]:
txn_raw["amount"].describe().round(2)

count    626169.00
mean       1852.85
std        2679.21
min      -47261.19
25%         465.48
50%        1065.48
75%        2242.89
max      164155.52
Name: amount, dtype: float64

Минимум отрицательный (-47261.19). Проверяем, у каких операций сумма меньше нуля.

In [25]:
print("операций с отрицательной суммой:", (txn_raw["amount"] < 0).sum(), "из", len(txn_raw))
print("из них со статусом возврат:", ((txn_raw["amount"] < 0) & (txn_raw["status"] == "возврат")).sum())
print("всего возвратов:", (txn_raw["status"] == "возврат").sum())

операций с отрицательной суммой: 7625 из 626169
из них со статусом возврат: 7625
всего возвратов: 7625


Все 7625 отрицательных сумм – возвраты, и все возвраты записаны с минусом. Возврат уменьшает траты клиента. Еще в `amount` вместе лежат и рубли, и евро, и доллары, поэтому `describe()` по всему столбцу дает только общее представление.

По описанию `monthly_income` – ежемесячный доход в рублях.

In [26]:
clients_raw["monthly_income"].describe().round(0)

count       5204.0
mean       95083.0
std        84636.0
min         -999.0
25%        52600.0
50%        72900.0
75%       108400.0
max      1135600.0
Name: monthly_income, dtype: float64

Минимум -999, а отрицательного дохода не бывает. Смотрим подробнее на маленькие значения.

In [27]:
income = clients_raw["monthly_income"]
ranges = pd.cut(income, [-1000, 0, 1000, 10000, income.max()],
                labels=["0 и меньше", "от 0 до 1000", "от 1000 до 10000", "больше 10000"])
ranges.value_counts(sort=False, dropna=False)

monthly_income
0 и меньше            54
от 0 до 1000         172
от 1000 до 10000       0
больше 10000        4978
NaN                  486
Name: count, dtype: int64

Таблица показывает, что есть две подозрительные группы: 54 строки с доходом 0 и меньше и 172 строки с доходом меньше 1000 рублей в месяц, причем между 1000 и 10000 нет ни одного значения. Смотрим, какие значения в каждой группе.

In [28]:
print("разные значения среди дохода 0 и меньше:", income[income <= 0].unique().astype(int))

разные значения среди дохода 0 и меньше: [-999]


Во всех 54 строках доход равен ровно -999. Если бы это были ошибки ввода, числа были бы разные, а одно и то же круглое значение у всех – это условный код «нет данных». Значит это не доход, а еще один скрытый пропуск.

In [29]:
income[(income > 0) & (income < 1000)].describe().round(1)

count    172.0
mean      96.5
std       77.1
min       17.9
25%       57.3
50%       77.4
75%      111.6
max      639.1
Name: monthly_income, dtype: float64

Эти 172 значения лежат от 17.9 до 639.1, медиана 77.4. Доход 77 рублей в месяц не бывает, а медиана всего дохода 72900 рублей, то есть 72.9 тысячи. Похоже, у этих клиентов доход записан в тысячах рублей, а не в рублях, как сказано в описании. На это же указывает пустой промежуток от 1000 до 10000.

По описанию `satisfaction_score` – оценка от 1 до 5.

In [30]:
clients_raw["satisfaction_score"].value_counts(dropna=False).sort_index()

satisfaction_score
1.0     298
2.0     464
3.0     447
4.0     471
5.0     651
NaN    3359
Name: count, dtype: int64

Все оценки от 1 до 5, других значений нет. Пропусков 3359 из 5690, то есть оценки нет больше чем у половины строк.

Теперь рассмотрим даты. Форматы в описании не указаны. Разбираем каждую дату клиентов с явным форматом и считаем, сколько значений не разобралось. 

In [31]:
date_formats = {"birth_date": "%Y-%m-%d", "registration_date": "%d.%m.%Y",
                "card_open_date": "%d.%m.%Y", "app_last_login": "%Y-%m"}
for col, fmt in date_formats.items():
    d = pd.to_datetime(clients_raw[col], format=fmt, errors="coerce")
    print(col, "| не разобралось:", (d.isna() & clients_raw[col].notna()).sum(),
          "| от", d.min().date(), "до", d.max().date())

birth_date | не разобралось: 0 | от 1936-07-23 до 2002-04-27
registration_date | не разобралось: 0 | от 2010-02-16 до 2025-03-10
card_open_date | не разобралось: 0 | от 2016-01-01 до 2025-05-16
app_last_login | не разобралось: 0 | от 2024-01-01 до 2025-06-01


Все даты клиентов разобрались, но записаны в разных форматах. Все даты не позже 30.06.2025, на которую составлен справочник.

В `txn_datetime` у части операций есть время, а у части только дата. Разбираем с форматом `ISO8601`, он понимает оба варианта, и сверяем с периодом из описания.

In [32]:
txn_dt = pd.to_datetime(txn_raw["txn_datetime"], format="ISO8601", errors="coerce")
no_time = txn_raw["txn_datetime"].str.len() == 10
print("не разобралось:", txn_dt.isna().sum())
print("операций без времени:", no_time.sum(), "из", len(txn_raw))
print("самая ранняя:", txn_dt.min(), "| самая поздняя:", txn_dt.max())
print("раньше 2024-01-01:", (txn_dt < "2024-01-01").sum())
print("позже 2025-06-30:", (txn_dt >= "2025-07-01").sum())

не разобралось: 0
операций без времени: 50087 из 626169
самая ранняя: 2023-11-24 09:00:10 | самая поздняя: 2026-08-04 12:27:10
раньше 2024-01-01: 6
позже 2025-06-30: 81


Все даты разобрались. 50087 операций из 626169 записаны без времени. По описанию данные охватывают 2024-01-01 – 2025-06-30, но есть 6 операций раньше и 81 позже, вплоть до августа 2026 года, то есть позже даты, на которую составлен справочник. Описание с этим не совпадает.

Проверяем, не сосредоточены ли операции без времени в каком-то одном месяце. Доля считается от всех операций месяца.

In [33]:
(no_time.groupby(txn_dt.dt.to_period("M")).mean() * 100).round(1).loc["2024-01":"2025-06"]

txn_datetime
2024-01    7.7
2024-02    8.2
2024-03    8.2
2024-04    7.9
2024-05    8.0
2024-06    8.1
2024-07    7.8
2024-08    7.9
2024-09    8.1
2024-10    8.1
2024-11    7.9
2024-12    8.2
2025-01    7.8
2025-02    7.9
2025-03    8.0
2025-04    8.2
2025-05    7.9
2025-06    7.8
Freq: M, Name: txn_datetime, dtype: float64

В каждом месяце периода без времени от 7.7% до 8.2% операций, то есть время пропадает равномерно, а не в каком-то одном месяце. Дата у этих операций есть, поэтому для подсчета по месяцам это не мешает.

Дальше рассмотрим логику между полями. Проверяем то, что должно выполняться по смыслу: карту не могли открыть раньше прихода в банк, клиент не мог прийти в банк слишком рано по возрасту (свою карту можно получить с 14 лет), последний вход в приложение не может быть раньше регистрации, а у клиентов без приложения не должно быть даты входа.

In [34]:
birth = pd.to_datetime(clients_raw["birth_date"], format="%Y-%m-%d")
reg = pd.to_datetime(clients_raw["registration_date"], format="%d.%m.%Y")
card_open = pd.to_datetime(clients_raw["card_open_date"], format="%d.%m.%Y")
last_login = pd.to_datetime(clients_raw["app_last_login"], format="%Y-%m")
age_at_reg = (reg - birth).dt.days / 365.25

print("карта открыта раньше прихода в банк:", (card_open < reg).sum())
print("пришли в банк младше 14 лет:", (age_at_reg < 14).sum())
print("последний вход в приложение раньше месяца регистрации:", (last_login < reg.dt.to_period("M").dt.to_timestamp()).sum())
print("нет приложения, но есть дата входа:", ((clients_raw["has_mobile_app"] == "Нет") & last_login.notna()).sum())
print("есть приложение, но нет даты входа:", ((clients_raw["has_mobile_app"] == "Да") & last_login.isna()).sum())

карта открыта раньше прихода в банк: 0
пришли в банк младше 14 лет: 4


последний вход в приложение раньше месяца регистрации: 25
нет приложения, но есть дата входа: 0
есть приложение, но нет даты входа: 0


Карт, открытых раньше регистрации, нет. Но 4 клиента пришли в банк младше 14 лет, а у 25 клиентов последний вход в приложение раньше, чем они вообще стали клиентами банка. Пропуски в `app_last_login` есть только у клиентов без приложения.

По описанию `transactions.csv` – операции по картам. Но в пояснении к каналам сказано, что `СБП` – перевод по Системе быстрых платежей, а `Онлайн-банк` – операция в интернет-банке, то есть это не оплата картой. Смотрим, сколько операций в каждом канале. Доля считается от всех 626169 операций.

In [35]:
channel_counts = txn_raw["channel"].value_counts()
pd.DataFrame({"операций": channel_counts, "доля, %": (channel_counts / len(txn_raw) * 100).round(1)})

,операций,"доля, %"
channel,,
Карта,321575,51.4
СБП,107549,17.2
Токен,102000,16.3
Онлайн-банк,95045,15.2


Оплаты картой – это `Карта` (пластик) и `Токен` (телефон или часы). А `СБП` и `Онлайн-банк` занимают 17.2% и 15.2% операций, то есть в файле есть не только операции по картам, как сказано в описании.

### 2.4. Связь между таблицами

Проверяем, совпадают ли клиенты в двух таблицах. Объединяем списки `client_id` с `indicator=True`: он показывает, где нашелся клиент – только в операциях (`left_only`), только в справочнике (`right_only`) или в обеих таблицах (`both`).

In [36]:
ids = txn_raw[["client_id"]].drop_duplicates().merge(
    clients_raw[["client_id"]].drop_duplicates(), how="outer", indicator=True)
ids["_merge"].value_counts()

_merge
both          5625
left_only      365
right_only      10
Name: count, dtype: int64

У 5625 клиентов есть и операции, и запись в справочнике. 365 клиентов есть в операциях, но их нет в справочнике, а 10 клиентов из справочника не сделали ни одной операции.

Смотрим на клиентов не из справочника: сколько у них операций и когда была их последняя операция.

In [37]:
unknown_ids = ids.loc[ids["_merge"] == "left_only", "client_id"]
unknown = txn_raw["client_id"].isin(unknown_ids)
print("операций у клиентов не из справочника:", unknown.sum(), "из", len(txn_raw))
txn_dt[unknown].groupby(txn_raw.loc[unknown, "client_id"]).max().dt.to_period("Q").value_counts().sort_index()

операций у клиентов не из справочника: 32642 из 626169


txn_datetime
2024Q3      1
2024Q4    179
2025Q1    181
2025Q2      3
2025Q3      1
Freq: Q-DEC, Name: count, dtype: int64

У этих 365 клиентов 32642 операции. Последняя операция почти у всех в конце 2024 – начале 2025 года: у 179 в 4 квартале 2024 и у 181 в 1 квартале 2025. Справочник – это действующие клиенты на 30.06.2025, поэтому похоже, что это клиенты, которые ушли из банка. Операции у них есть в базовом периоде, а потом прекращаются, поэтому их уход может выглядеть как падение активности.

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

Сильнее всего мешают три вещи. В операциях есть 365 клиентов, которых нет в справочнике действующих клиентов, у них 32642 операции из 626169. Последняя операция у них почти у всех в конце 2024 – начале 2025 года, поэтому похоже, что они ушли из банка, и если их не отделить, их уход будет выглядеть как падение активности. Операции через `СБП` и `Онлайн-банк` (17.2% и 15.2% от 626169 операций, вместе около трети) не являются оплатой картой, хотя файл описан как операции по картам, поэтому от выбора каналов зависит сама метрика. Наконец, 1500 операций записаны дважды – это завышает активность.

Мешают группировать, но легко исправляются: города записаны 17 способами вместо 6, категорий 12 вместо 9, у 40 строк клиентов есть полные копии, а у 15 клиентов две разные записи с разными городом или сегментом.

Касаются отдельных полей: доход пустой у 486 строк из 5690, равен -999 у 54 и похож на записанный в тысячах у 172, источник дохода `Не указан` у 1544, оценки качества нет у 3359. Это важно, только если эти поля войдут в правила групп. Еще есть операции в валюте (1283 в `EUR` и 1250 в `USD`) и операции вне периода из описания (6 раньше и 81 позже) – их мало.

Мелочи, на выводы почти не влияют: 1645 операций, где код торговой точки не совпадает с категорией, 4 клиента, которые пришли в банк младше 14 лет, и 25 клиентов со входом в приложение раньше регистрации.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

Все исправления делаем на копиях, сырые данные не меняем так мы будем сравнивать, что было и что стало. Заводим журнал: каждое решение, которое меняет данные, записываем функцией `log_change`.

In [38]:
clients = clients_raw.copy()
txn = txn_raw.copy()

log_rows = []


def log_change(field, issue, scope, n_rows, value_before, action, value_after, rationale):
    log_rows.append({"field": field, "issue": issue, "scope": scope, "n_rows": n_rows,
                     "value_before": value_before, "action": action,
                     "value_after": value_after, "rationale": rationale})

Даты клиентов записаны в разных форматах. Переводим их в тип дата с теми же явными форматами, что что задали в 2.3. `app_last_login` хранит только месяц, поэтому превращается в первое число месяца.

In [39]:
for col, fmt in date_formats.items():
    clients[col] = pd.to_datetime(clients[col], format=fmt)

log_change("birth_date, registration_date, card_open_date, app_last_login",
           "даты записаны текстом в трех разных форматах", "все строки clients", len(clients),
           "'1972-01-20', '21.03.2022', '2024-04'", "исправлено", "тип дата",
           "форматы заданы явно, все значения разобрались без ошибок")
clients[list(date_formats)].dtypes

birth_date           datetime64[us]
registration_date    datetime64[us]
card_open_date       datetime64[us]
app_last_login       datetime64[us]
dtype: object

Дату операции тоже переводим.

In [40]:
txn["txn_datetime"] = pd.to_datetime(txn["txn_datetime"], format="ISO8601")

log_change("txn_datetime", "дата записана текстом, у части операций нет времени", "все строки transactions",
           len(txn), "'2024-01-01 11:56:08', '2024-01-01'", "исправлено", "тип дата-время, без времени -> 00:00:00",
           "формат ISO8601 понимает оба варианта, все значения разобрались (проверено в 2.3)")
print("тип txn_datetime:", txn["txn_datetime"].dtype)

тип txn_datetime: datetime64[us]


Доход. Значение -999 скрытый пропуск, а не доход, поэтому заменяем его на пропуск.

In [41]:
is_code = clients["monthly_income"] == -999
clients.loc[is_code, "monthly_income"] = np.nan

log_change("monthly_income", "код -999 вместо дохода", "monthly_income == -999", is_code.sum(),
           -999, "заменено на пропуск", "NaN",
           "доход меньше нуля и одинаковый у всех 54 строк – это скрытый пропуск")
print("заменено на пропуск:", is_code.sum())

заменено на пропуск: 54


Доходы меньше 1000 рублей в месяц выглядят как записанные в тысячах: после них сразу идет большой разрыв до обычных доходов. Умножаем их на 1000 и проверяем, станут ли они похожи на доходы остальных клиентов.

In [42]:
in_thousands = (clients["monthly_income"] > 0) & (clients["monthly_income"] < 1000)
clients.loc[in_thousands, "monthly_income"] = clients.loc[in_thousands, "monthly_income"] * 1000

log_change("monthly_income", "доход записан в тысячах рублей", "0 < monthly_income < 1000", in_thousands.sum(),
           "17.9 – 639.1", "исправлено", "17900 – 639100",
           "между 1000 и 10000 нет ни одного значения, медиана 77.4 против 72900 у остальных – тот же доход в тысячах")
print("умножено на 1000:", in_thousands.sum())
clients.loc[in_thousands, "monthly_income"].describe().round(0)

умножено на 1000: 172


count       172.0
mean      96451.0
std       77055.0
min       17900.0
25%       57275.0
50%       77350.0
75%      111575.0
max      639100.0
Name: monthly_income, dtype: float64

После умножения эти доходы лежат от 17900 до 639100, медиана 77350 как и у остальных клиентов Предположение про тысячи подтверждается.

### 3.2. Категориальные признаки

Приводим названия городов к одному виду.

In [43]:
city_map = {
    "г. москва": "москва",
    "спб": "санкт-петербург",
    "санкт петербург": "санкт-петербург",
    "г. санкт-петербург": "санкт-петербург",
    "г. новосибирск": "новосибирск",
    "г.екатеринбург": "екатеринбург",
    "н. новгород": "нижний новгород",
}
city_before = clients["city"].copy()
clients["city"] = clients["city"].str.strip().str.lower().replace(city_map).str.title()

log_change("city", "один город записан по-разному", "все строки clients", (city_before != clients["city"]).sum(),
           "' москва', 'МОСКВА', 'г. Москва', 'СПб'", "исправлено", "6 городов: 'Москва', 'Санкт-Петербург' и т.д.",
           "это разные написания одного города, смысл не меняется")
print("изменено строк:", (city_before != clients["city"]).sum())
clients["city"].value_counts()

изменено строк: 1600


city
Москва             1661
Санкт-Петербург    1010
Новосибирск         905
Екатеринбург        713
Казань              705
Нижний Новгород     696
Name: count, dtype: int64

Объединяем одинаковые категории, записанные по-разному.

In [44]:
category_map = {
    "Кафе/рестораны": "Кафе и рестораны",
    "кафе и рестораны": "Кафе и рестораны",
    "Связь и Интернет": "Связь и интернет",
}
category_changed = txn["category"].isin(list(category_map))
txn["category"] = txn["category"].replace(category_map)

log_change("category", "одна категория записана по-разному", "category in " + str(list(category_map)),
           category_changed.sum(), "'Кафе/рестораны', 'кафе и рестораны', 'Связь и Интернет'", "исправлено",
           "'Кафе и рестораны', 'Связь и интернет'", "разные написания одной категории, у них один и тот же mcc_code")
print("изменено строк:", category_changed.sum(), "| категорий:", txn["category"].nunique())

изменено строк: 54017 | категорий: 9


Категорий стало 9. Несовпадения `mcc_code` с категорией оставляем как есть их у нас мало мало, а для анализа берем `category`. Это тоже решение, записываем его в журнал.

In [45]:
log_change("mcc_code", "код не совпадает с основной категорией", "mcc_code != основной код категории",
           mcc_mismatch.sum(), "например 'АЗС' с кодом 5411", "оставлено", "без изменений",
           "мало (1645 из 626169), в анализе используется category, что верно – код или категория – по данным не понять")

`income_source = Не указан` – скрытый пропуск, заменяем его на настоящий пропуск, чтобы он не считался отдельным источником дохода. 

In [46]:
not_given = clients["income_source"] == "Не указан"
clients.loc[not_given, "income_source"] = np.nan

log_change("income_source", "значение «Не указан» вместо источника дохода", "income_source == 'Не указан'",
           not_given.sum(), "Не указан", "заменено на пропуск", "NaN",
           "это отметка об отсутствии данных, а не источник дохода")
print("заменено на пропуск:", not_given.sum())

заменено на пропуск: 1544


`has_mobile_app` и `marketing_opt_in` – да/нет. Переводим их в `True` и `False`, так их будет удобнее считать и фильтровать.

In [47]:
for col in ["has_mobile_app", "marketing_opt_in"]:
    clients[col] = clients[col].map({"Да": True, "Нет": False})

log_change("has_mobile_app, marketing_opt_in", "да/нет записано текстом", "все строки clients", len(clients),
           "'Да', 'Нет'", "исправлено", "True, False", "смысл не меняется, так проще фильтровать")
clients[["has_mobile_app", "marketing_opt_in"]].value_counts(dropna=False)

has_mobile_app  marketing_opt_in
True            True                3103
False           True                1919
True            False                422
False           False                246
Name: count, dtype: int64

Пустых значений не появилось, значит других вариантов, кроме `Да` и `Нет`, не было.

У loyalty_tier есть порядок. Задаем его явно, иначе pandas отсортирует уровни по алфавиту.

In [48]:
loyalty_order = ["Базовый", "Серебро", "Золото", "Платина"]
clients["loyalty_tier"] = pd.Categorical(clients["loyalty_tier"], categories=loyalty_order, ordered=True)

log_change("loyalty_tier", "порядковая шкала записана текстом", "все строки clients", len(clients),
           "'Базовый', 'Серебро', 'Золото', 'Платина'", "исправлено", "упорядоченная категория",
           "уровни лояльности упорядочены, порядок задан явно")
clients["loyalty_tier"].value_counts(sort=False)

loyalty_tier
Базовый    2858
Серебро    1505
Золото     1013
Платина     314
Name: count, dtype: int64

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

Сколько пропусков осталось после 3.1 и 3.2. Сюда уже входят скрытые пропуски: -999 в доходе и `Не указан` в источнике дохода. 

In [49]:
missing = clients.isna().sum()
missing = missing[missing > 0].to_frame("пропусков")
missing["из"] = len(clients)
missing["доля, %"] = (missing["пропусков"] / len(clients) * 100).round(1)
print("пропусков в transactions:", txn.isna().sum().sum())
missing

пропусков в transactions: 0


,пропусков,из,"доля, %"
income_source,1544,5690,27.1
monthly_income,540,5690,9.5
app_last_login,2165,5690,38.0
satisfaction_score,3359,5690,59.0


Чтобы понять механизм пропуска, сравниваем долю пропусков в разных группах клиентов. Если пропуски случайные, доля будет примерно одинаковой во всех группах, а если зависит от группы – пропуск связан с этим признаком. Доля считается внутри каждой группы: например, от всех клиентов массового сегмента.

In [50]:
missing_cols = ["monthly_income", "income_source", "satisfaction_score"]
is_missing = clients[missing_cols].isna()
display((is_missing.groupby(clients["segment"]).mean() * 100).round(1))
(is_missing.groupby(clients["loyalty_tier"]).mean() * 100).round(1)

,monthly_income,income_source,satisfaction_score
segment,,,
Private,6.6,29.2,31.9
Зарплатный,5.1,4.9,59.8
Массовый,12.0,40.1,64.6
Премиальный,9.8,24.0,47.4


,monthly_income,income_source,satisfaction_score
loyalty_tier,,,
Базовый,9.8,31.0,68.1
Серебро,8.8,23.1,66.5
Золото,10.1,23.7,29.7
Платина,8.3,22.3,35.7


`monthly_income`: доход не указан у 12.0% массового сегмента и только у 5.1% зарплатного. Зарплату зарплатных клиентов банк видит сам, а остальные указывают доход со своих слов и чаще его не называют. Пропуск зависит от сегмента, который известен, – это MAR (случайный при условии других признаков).

`income_source`: та же картина, но сильнее – 40.1% у массового сегмента и 4.9% у зарплатного. Тоже MAR.

`satisfaction_score`: оценки нет у 68.1% клиентов уровня `Базовый` и 66.5% `Серебро`, но только у 29.7% `Золото` и 35.7% `Платина`. Опрос чаще проходят клиенты с высоким уровнем лояльности, поэтому это MAR. Может быть и MNAR, если недовольные клиенты чаще отказываются от опроса, но по данным это не проверить.

`app_last_login`: пропуск ровно у клиентов без приложения. Это не потеря данных, а неприменимое значение, так как входить некуда.

Наше решение – это ничего не заполнять. Активность будем считать по операциям, а в них пропусков нет. Заполнять доход медианой нельзя пропуски зависят от сегмента, а одно значение для всех сожмет разброс и сдвинет доход массового сегмента. `satisfaction_score` – порядковая шкала, и заполнять ее средним бессмысленно. Для групп эти поля используем только там, где значение известно. Решения записываем в журнал.

In [51]:
for col, mech, why in [
    ("monthly_income", "MAR: чаще не указан у массового сегмента",
     "в метрике не участвует, заполнение одним значением сожмет разброс и сдвинет доход массового сегмента"),
    ("income_source", "MAR: чаще не указан у массового сегмента",
     "в метрике не участвует, заполнить по данным нечем"),
    ("satisfaction_score", "MAR или MNAR: чаще нет у уровней Базовый и Серебро",
     "порядковая шкала, среднее для нее не считается, ответившие не представляют всех клиентов"),
    ("app_last_login", "неприменимо: у клиента нет приложения",
     "значения не может быть, пропуск сам по себе означает отсутствие приложения"),
]:
    log_change(col, "пропуски, механизм " + mech, col + " is NaN", clients[col].isna().sum(),
               "NaN", "оставлено", "NaN", why)
print("записей в журнале:", len(log_rows))

записей в журнале: 14


### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

**Шаг 1. Обнаружение.** Начинаем с сумм операций. Берем только рублевые суммы больше нуля: операции в валюте в другом масштабе, а возвраты отрицательные по смыслу. Ищем кандидатов в выбросы правилом межквартильного размаха (МКР) с множителями 1.5 и 3 и смотрим, сколько операций выше 99.9-го процентиля.

In [52]:
rub = txn.loc[(txn["currency"] == "RUB") & (txn["amount"] > 0), "amount"]
q1, q3 = rub.quantile([0.25, 0.75])
iqr = q3 - q1
print("операций:", len(rub), "| Q1:", round(q1, 2), "| медиана:", round(rub.median(), 2), "| Q3:", round(q3, 2))
for k in [1.5, 3]:
    upper = q3 + k * iqr
    print(f"граница Q3 + {k} МКР: {upper:.2f} | операций выше: {(rub > upper).sum()} ({(rub > upper).mean() * 100:.1f}% от {len(rub)})")
p999 = rub.quantile(0.999)
print(f"99.9-й процентиль: {p999:.2f} | операций выше: {(rub > p999).sum()}")

операций: 616034 | Q1: 490.25 | медиана: 1090.26 | Q3: 2274.34
граница Q3 + 1.5 МКР: 4950.49 | операций выше: 50213 (8.2% от 616034)
граница Q3 + 3 МКР: 7626.63 | операций выше: 21498 (3.5% от 616034)
99.9-й процентиль: 26160.75 | операций выше: 617


Правило 1.5 МКР отмечает 8.2% из 616034 рублевых операций – слишком много, чтобы считать их выбросами. Просто у сумм естественный длинный хвост: большинство покупок небольшие, но покупки на несколько тысяч тоже обычны. Даже с множителем 3 остается 3.5% из них. Смотрим на самые крупные операции.

In [53]:
txn.nlargest(10, "amount")[["client_id", "txn_datetime", "amount", "currency", "category", "channel", "status"]]

,client_id,txn_datetime,amount,currency,category,channel,status
440504,C004150,2024-12-23 18:57:12,164155.52,RUB,Развлечения и досуг,Карта,успешно
623569,C003102,2025-06-28 14:45:43,86726.59,RUB,Одежда и обувь,Онлайн-банк,успешно
175946,C000882,2024-06-05 07:24:09,80290.80,RUB,Одежда и обувь,Карта,отклонено
602330,C000973,2025-06-07 15:09:37,77680.72,RUB,Одежда и обувь,Карта,успешно
134463,C002913,2024-04-28 21:45:05,73878.39,RUB,Одежда и обувь,Карта,отклонено
18868,C002086,2024-01-21 17:09:48,70114.15,RUB,Одежда и обувь,СБП,отклонено
300279,C000309,2024-09-10 13:44:12,69185.15,RUB,Одежда и обувь,Карта,отклонено
271513,C005622,2024-08-19 18:37:42,66992.31,RUB,Одежда и обувь,Токен,успешно
136001,C004018,2024-04-30 13:55:51,63990.13,RUB,Одежда и обувь,Онлайн-банк,успешно
79114,C005655,2024-03-16 11:48:47,63633.76,RUB,Одежда и обувь,Онлайн-банк,успешно


Самая крупная операция – 164155.52 рубля в категории `Развлечения и досуг`, почти вдвое больше следующей (86726.59). Остальные крупные суммы – покупки одежды до 87 тысяч, часть из них отклонена. Такие покупки правдоподобны, признаков ошибки ввода вроде лишнего нуля или кода нет.

Теперь доход клиентов, уже после исправлений в 3.1. Здесь берем множитель 3, чтобы найти только далекие значения, и смотрим, в каких сегментах они оказались.

In [54]:
income = clients["monthly_income"]
inc_q1, inc_q3 = income.quantile([0.25, 0.75])
inc_upper = inc_q3 + 3 * (inc_q3 - inc_q1)
print(f"граница Q3 + 3 МКР: {inc_upper:.0f} | клиентов выше: {(income > inc_upper).sum()} из {income.notna().sum()} с известным доходом")
print("медианный доход по сегментам:", income.groupby(clients["segment"]).median().to_dict())
clients.loc[income > inc_upper, "segment"].value_counts()

граница Q3 + 3 МКР: 279200 | клиентов выше: 196 из 5150 с известным доходом
медианный доход по сегментам: {'Private': 334300.0, 'Зарплатный': 76950.0, 'Массовый': 61050.0, 'Премиальный': 148600.0}


segment
Private        146
Премиальный     48
Зарплатный       1
Массовый         1
Name: count, dtype: int64

Выше границы 279200 оказались 196 клиентов из 5150 с известным доходом, и почти все они из сегментов `Private` (146) и `Премиальный` (48). Медианный доход в `Private` – 334300, так что высокий доход для этих клиентов – норма, а не ошибка.

**Шаг 2. Решение.** Ничего не удаляем и не ограничиваем: ошибку ни в суммах, ни в доходе доказать нельзя, а крупные значения правдоподобны. Операции выше 99.9-го процентиля отмечаем флагом `amount_extreme`, чтобы потом проверить, не меняются ли выводы без них. Активность будем считать устойчивыми мерами (число операций, медианы), на которые крупные суммы почти не влияют. Записываем решения в журнал.

In [55]:
txn["amount_extreme"] = (txn["currency"] == "RUB") & (txn["amount"] > p999)

log_change("amount", "суммы выше 99.9-го процентиля рублевых операций", f"currency == 'RUB' and amount > {p999:.2f}",
           txn["amount_extreme"].sum(), f"от {p999:.0f} до {rub.max():.0f}", "оставлено, добавлен флаг amount_extreme",
           "без изменений", "ошибку доказать нельзя, крупные покупки правдоподобны; флаг нужен для проверки чувствительности выводов")
log_change("monthly_income", "доход выше Q3 + 3 МКР", f"monthly_income > {inc_upper:.0f}", (income > inc_upper).sum(),
           f"от {inc_upper:.0f} до {income.max():.0f}", "оставлено", "без изменений",
           "почти все такие клиенты из сегментов Private и Премиальный, для них высокий доход нормален")
print("отмечено флагом amount_extreme:", txn["amount_extreme"].sum())

отмечено флагом amount_extreme: 617


### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

Сначала справочник клиентов. Полные копии строк удаляем. Клиентов, у которых две разные записи, оставляем одной записью, но помечаем флагом `conflict`: их город или сегмент неизвестен, поэтому в кампанию их не возьмем.

In [56]:
n_clients_before = len(clients)
full_copies = clients.duplicated()
clients = clients[~full_copies]

conflict_ids = clients.loc[clients["client_id"].duplicated(keep=False), "client_id"].unique()
clients["conflict"] = clients["client_id"].isin(conflict_ids)
n_conflict_extra = clients["client_id"].duplicated().sum()
clients = clients.drop_duplicates("client_id", keep="first")

log_change("все поля clients", "полные копии строк", "строки, полностью совпадающие с другой строкой",
           full_copies.sum(), "две одинаковые строки", "удалено", "одна строка",
           "техническое задвоение выгрузки, данные не теряются")
log_change("city, segment", "у одного client_id две разные записи", "client_id in conflict_ids",
           n_conflict_extra, "две записи с разными городом или сегментом", "оставлена первая запись, флаг conflict",
           "одна строка на клиента", "какая запись верная, по данным не понять; такие клиенты не идут в кампанию")
print("строк было:", n_clients_before, "| удалено полных копий:", full_copies.sum(),
      "| удалено вторых записей:", n_conflict_extra, "| осталось:", len(clients))
print("клиентов с флагом conflict:", clients["conflict"].sum(), "| client_id уникальны:", clients["client_id"].is_unique)

строк было: 5690 | удалено полных копий: 41 | удалено вторых записей: 14 | осталось: 5635
клиентов с флагом conflict: 14 | client_id уникальны: True


Полных копий оказалось 41, а не 40, как в аудите: после приведения городов к одному виду у клиента `C002520` (`Нижний Новгород` и `Н. Новгород`) две записи стали одинаковыми. Поэтому клиентов с разными записями осталось 14, их помечаем флагом `conflict`. Теперь в справочнике 5635 строк, по одной на каждого клиента.

Теперь операции. Фильтры применяем по очереди и после каждого записываем, сколько строк осталось.

In [57]:
funnel = [("все операции", len(txn))]

has_time = txn_raw["txn_datetime"].str.len() > 10
before = len(txn)
txn = (txn.assign(has_time=has_time).sort_values("has_time", ascending=False)
          .drop_duplicates("txn_id").drop(columns="has_time").sort_index())
log_change("txn_id", "одна операция записана дважды", "повторы txn_id", before - len(txn),
           "две строки с одним txn_id", "удалено", "одна строка, из пары оставлена запись со временем",
           "это одна и та же операция (проверено в 2.2), иначе она посчитается дважды")
funnel.append(("без задвоенных операций", len(txn)))

in_period = (txn["txn_datetime"] >= "2024-01-01") & (txn["txn_datetime"] < "2025-07-01")
log_change("txn_datetime", "дата вне периода 2024-01-01 – 2025-06-30", "txn_datetime < 2024-01-01 или >= 2025-07-01",
           (~in_period).sum(), "2023-11-24 ... 2026-08-04", "удалено", "только 2024-01-01 – 2025-06-30",
           "не попадают ни в базовое, ни в оцениваемое окно, часть дат позже выгрузки справочника")
txn = txn[in_period]
funnel.append(("в периоде 2024-01-01 – 2025-06-30", len(txn)))

known = txn["client_id"].isin(clients["client_id"])
log_change("client_id", "клиента нет в справочнике действующих клиентов", "client_id not in clients",
           (~known).sum(), f"{txn.loc[~known, 'client_id'].nunique()} клиентов", "удалено", "только клиенты из справочника",
           "похоже на ушедших клиентов (последние операции в конце 2024 – начале 2025), вернуть их кампанией нельзя")
txn = txn[known]
funnel.append(("только клиенты из справочника", len(txn)))

txn_ok = txn[txn["status"] == "успешно"]
log_change("status", "отклоненные операции и возвраты", "status != 'успешно'", len(txn) - len(txn_ok),
           "отклонено, возврат", "исключено из расчета активности", "только status == 'успешно'",
           "отклонение – попытка, а не пользование картой, возврат – отмена покупки; доля отказов стабильна по месяцам")
funnel.append(("только успешные (txn_ok)", len(txn_ok)))

funnel = pd.DataFrame(funnel, columns=["шаг", "осталось операций"])
funnel["убрано"] = (funnel["осталось операций"].shift() - funnel["осталось операций"]).fillna(0).astype(int)
funnel

,шаг,осталось операций,убрано
0,все операции,626169,0
1,без задвоенных операций,624669,1500
2,в периоде 2024-01-01 – 2025-06-30,624582,87
3,только клиенты из справочника,592028,32554
4,только успешные (txn_ok),573137,18891


Из 626169 операций для расчета активности осталось 573137. Больше всего ушло на клиентах, которых нет в справочнике (32554 операции), задвоенных операций было 1500, вне периода – 87. Отклоненные операции и возвраты (18891) не удаляем из `txn`, а только не включаем в `txn_ok`, по которой будем считать активность.

Проверяем, что, исключая отклоненные операции, мы не прячем причину падения: если бы в оцениваемом периоде отказов стало больше, это само могло бы объяснять снижение активности. Доля считается от всех операций месяца.

In [58]:
(txn.groupby(txn["txn_datetime"].dt.to_period("M"))["status"].value_counts(normalize=True).unstack() * 100).round(1)

status,возврат,отклонено,успешно
txn_datetime,,,
2024-01,1.2,2.1,96.8
2024-02,1.2,2.2,96.6
2024-03,1.1,2.0,96.8
2024-04,1.3,2.0,96.7
2024-05,1.3,2.2,96.5
2024-06,1.2,2.0,96.7
2024-07,1.1,2.0,96.9
2024-08,1.2,1.9,97.0
2024-09,1.3,1.9,96.8


Доля отклоненных операций держится от 1.8% до 2.2% в каждом месяце, в 2025 году она не выросла. Значит отказы не объясняют падение активности и исключать их из расчета можно.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [59]:
cleaning_log = pd.DataFrame(log_rows, columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])
print("решений в журнале:", len(cleaning_log))
cleaning_log

решений в журнале: 22


,field,issue,scope,n_rows,value_before,action,value_after,rationale
0,"birth_date, registration_date, card_open_date,...",даты записаны текстом в трех разных форматах,все строки clients,5690,"'1972-01-20', '21.03.2022', '2024-04'",исправлено,тип дата,"форматы заданы явно, все значения разобрались ..."
1,txn_datetime,"дата записана текстом, у части операций нет вр...",все строки transactions,626169,"'2024-01-01 11:56:08', '2024-01-01'",исправлено,"тип дата-время, без времени -> 00:00:00","формат ISO8601 понимает оба варианта, все знач..."
2,monthly_income,код -999 вместо дохода,monthly_income == -999,54,-999,заменено на пропуск,NaN,доход меньше нуля и одинаковый у всех 54 строк...
3,monthly_income,доход записан в тысячах рублей,0 < monthly_income < 1000,172,17.9 – 639.1,исправлено,17900 – 639100,"между 1000 и 10000 нет ни одного значения, мед..."
4,city,один город записан по-разному,все строки clients,1600,"' москва', 'МОСКВА', 'г. Москва', 'СПб'",исправлено,"6 городов: 'Москва', 'Санкт-Петербург' и т.д.","это разные написания одного города, смысл не м..."
5,category,одна категория записана по-разному,"category in ['Кафе/рестораны', 'кафе и рестора...",54017,"'Кафе/рестораны', 'кафе и рестораны', 'Связь и...",исправлено,"'Кафе и рестораны', 'Связь и интернет'","разные написания одной категории, у них один и..."
6,mcc_code,код не совпадает с основной категорией,mcc_code != основной код категории,1645,например 'АЗС' с кодом 5411,оставлено,без изменений,"мало (1645 из 626169), в анализе используется ..."
7,income_source,значение «Не указан» вместо источника дохода,income_source == 'Не указан',1544,Не указан,заменено на пропуск,NaN,"это отметка об отсутствии данных, а не источни..."
8,"has_mobile_app, marketing_opt_in",да/нет записано текстом,все строки clients,5690,"'Да', 'Нет'",исправлено,"True, False","смысл не меняется, так проще фильтровать"
9,loyalty_tier,порядковая шкала записана текстом,все строки clients,5690,"'Базовый', 'Серебро', 'Золото', 'Платина'",исправлено,упорядоченная категория,"уровни лояльности упорядочены, порядок задан явно"


## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [60]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")